In [3]:
import json
import numpy as np
import pyvisa
import time
import hid # pip install hidapi
import skrf as rf
import matplotlib.pyplot as plt
import serial.tools.list_ports
from windfreak import SynthHD
from RsInstrument import RsInstrument, BinFloatFormat

rm = pyvisa.ResourceManager()
instruments = rm.list_resources()


In [6]:
spa = None
for instrument in instruments:
    if '7405' in instrument:
        spa = rm.open_resource(instrument) 
        break

In [7]:
print(spa.query('*IDN?'))

Hewlett-Packard, E7405A, SG45102671, A.14.04



In [20]:
spa_json = {}
spa_json['idn'] = spa.query('*IDN?')
spa_json['fstart']    = float(spa.query(':SENS1:FREQ:START?'))
spa_json['fstop']     = float(spa.query(':SENS1:FREQ:STOP?'))
spa_json['numpoints'] = int(spa.query(':SENS1:SWEep:POINts?'))
spa_json['rbw']      = float(spa.query(':SENS:BAND?'))
spa_json['vbw'] = float(spa.query(':SENS:BAND:VID?'))
spa_json['unit'] = spa.query(":UNIT:POW?").strip()
spa.write(":SYST:LOCAL")
spa_json

{'idn': 'Hewlett-Packard, E7405A, SG45102671, A.14.04\n',
 'fstart': 4000000000.0,
 'fstop': 8000000000.0,
 'numpoints': 401,
 'rbw': 1000000.0,
 'vbw': 3000.0,
 'unit': 'DBUV'}

In [11]:
frequencies = np.linspace(spa_json['fstart'],spa_json['fstop'],spa_json['numpoints'])
fghz = frequencies/1e9

In [22]:
spa.write("FORM REAL,32")
spa.write("INIT:CONT OFF")
spa.timeout = 30000
spa.write("INIT:IMM")
spa.query("*OPC?")
spa.timeout = 5000
trace_data = spa.query_binary_values("TRAC? TRACE1", datatype="f", is_big_endian=True)
spa.write("INIT:CONT ON")
spa.write(":SYST:LOCAL")
spa.write(":SYST:LOCAL")


13

In [23]:
trace_data

[5.518232626400277e-08,
 5.452554319163028e-08,
 5.2396227800954875e-08,
 5.4050534714633613e-08,
 5.0675726726012726e-08,
 5.4237542457258314e-08,
 5.215548526393832e-08,
 5.637674505010182e-08,
 5.327212448946739e-08,
 5.305178518710818e-08,
 5.0535899021042496e-08,
 5.089791699219859e-08,
 5.180837803209215e-08,
 5.0408065277451897e-08,
 5.1451731764018405e-08,
 5.125072632949923e-08,
 5.1784525112452684e-08,
 5.249283319130882e-08,
 5.0535899021042496e-08,
 5.184417872783342e-08,
 5.148728376980216e-08,
 4.940831033195536e-08,
 4.984252299777836e-08,
 4.764309835536551e-08,
 5.0699068054882446e-08,
 4.8921528161827155e-08,
 4.668743258662289e-08,
 4.879777293353982e-08,
 5.002648606478033e-08,
 5.0933085304905035e-08,
 4.6730452396559485e-08,
 4.7719954210379e-08,
 5.198762664804235e-08,
 4.662297570234841e-08,
 4.951080612158876e-08,
 5.196369201598827e-08,
 4.829475841461317e-08,
 4.6913733342535124e-08,
 4.75773234143162e-08,
 4.668743258662289e-08,
 4.911340312219181e-08,
 4.68

In [17]:
spa.write(":SYST:LOCAL")

13

In [18]:
spa.query(":UNIT:POW?")


'DBUV\n'

In [19]:
spa.query(":UNIT:POW?").strip()

'DBUV'

In [21]:
spa.write(":UNIT:POW W")


13